# eph_09 — Structural axes: does RT encoding align with LC's anatomical organization?

**Question.** LC is organized along several spatial gradients — waveform type,
transcriptomic identity, and projection target. `eph_04` established that RT
encoding is spatially non-random within LC. This notebook asks the sharper
question: **is the RT-encoding gradient the _same_ gradient as one of the
structural ones, or its own?**

Each organizational scheme is reduced to a single 3-D direction in CCF space —
the axis along which that property changes fastest — so all of them become
comparable objects: unit vectors with bootstrap uncertainty.

**Pipeline:**
1. Units via `data_loading`; per-unit RT encoding via `AnalysisSpec` / `fit_encoding`,
   registered in `PerUnitStatsRegistry` alongside `eph_01`–`eph_04`
2. Fit the **RT-encoding** spatial axis (`T_rt`) and its **baseline** control
   (`T_rt_bl`) — scalar feature, so OLS (`fit_spatial_axis_linear`)
3. Fit three **structural** axes — waveform features (CCA), MERFISH
   transcriptomics (CCA), retrograde tracing (LDA)
4. Pairwise direction comparison: angle, Wald *W*, χ² *p*, bootstrap *p*
5. Visualize: projected arrows with 95% confidence cones, and the bootstrap
   clouds in azimuth–elevation
6. Projection scatters — does a unit's position along a structural gradient
   predict how strongly it encodes RT?

All axis machinery lives in `spatial_axes.py`, shared with `eph_08` (which
projects onto the waveform axis but never fits the RT axis itself).

**Code Ocean only.** Needs `combined_unit_tbl.pkl`, `all_counts_df.parquet`, the
LC CCF mesh, the waveform CSV, the MERFISH `.h5ad`, and the retrograde CSV.
Run locally it prints skip messages and exits clean.

> **Degrades, never fakes.** Each structural axis sits behind its own
> `HAS_WAVEFORM` / `HAS_MERFISH` / `HAS_RETRO` flag. A missing asset drops that
> axis from every downstream comparison and figure; it is never substituted or
> imputed. Check the flags printed in §5 before reading §6–§8.

## 1. Setup

In [ ]:
%matplotlib inline
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from plotstyle import apply_style, OKABE_ITO, PALETTE, style_ax, save_fig
from spatial_axes import (
    bootstrap_spatial_axis_linear,
    bootstrap_spatial_axis_cca,
    bootstrap_spatial_axis_LDA,
    compare_bootstrap_directions,
    cone_half_angle,
    vectors_to_az_el,
    plot_projected_arrow_with_cone,
    plot_projection_scatter,
)

apply_style()

if Path("/root/capsule").exists():
    ENV     = "codeocean"
    SCRATCH = Path("/root/capsule/scratch")
    DATA    = Path("/root/capsule/data")
else:
    ENV     = "local"
    SCRATCH = Path("/Users/mib/Documents/Code/kinematics_analysis/data/for_local").parent
    DATA    = SCRATCH

IS_CO    = ENV == "codeocean"
FIG_DIR  = SCRATCH / "figures" / "eph_09_structural_axes"
SAVE_FIG = False
N_BOOT   = 2000

# Structural-axis availability. Each is set True only by its own loader in §5,
# so a missing asset removes that axis instead of failing the notebook.
HAS_WAVEFORM = HAS_MERFISH = HAS_RETRO = False

# CCF axis order and bregma reference (shared with eph_08 / spatial_encoding.py)
ml, ap, dv = 0, 1, 2
BREGMA_LPS_MM  = np.array([-5.7, 5.4, -0.45], dtype=float)
BREGMA_PIR_VOX = np.array([216, 18, 228], dtype=float)
CCF_RES_UM = 25.0
PLANES     = {"sag": [ap, dv], "hor": [ml, ap], "cor": [ml, dv]}
AXIS_LABEL = {ml: "ML (mm)", ap: "AP (mm)", dv: "DV (mm)"}

# One colorblind-safe hue per axis (Okabe-Ito, via plotstyle). The source
# notebook used the upstream palette (red / orange / green / purple / peach) to match
# his figures; swap these back if you need a side-by-side with those.
COLORS = {
    "RT encoding":      PALETTE["pos"],                 # vermillion
    "RT baseline":      PALETTE["baseline"],            # bluish green
    "Waveform (CCA)":   OKABE_ITO["blue"],
    "MERFISH (CCA)":    OKABE_ITO["reddish_purple"],
    "Retrograde (LDA)": OKABE_ITO["orange"],
}

print(f"ENV={ENV}")
if not IS_CO:
    print("eph_09 is Code Ocean only (needs combined_unit_tbl, all_counts_df, LC mesh, "
          "waveform CSV, MERFISH h5ad, retrograde CSV). Cells below will skip locally.")

## 2. Units, RT encoding, CCF coordinates

Session QC and unit QC come from `data_loading`; the per-unit RT statistics come
from the same `AnalysisSpec` / `fit_encoding` path as `eph_01`, so `T_rt` here is
the identical quantity `eph_01`–`eph_04` report, and the results land in a
`PerUnitStatsRegistry` rather than being recomputed ad hoc.

`T_rt` is the response-window t-statistic of `spike_count ~ 1 + z(log RT)`;
`T_rt_bl` is the same regression in the pre-cue baseline window, and acts as the
control axis — if the baseline axis points the same way as the response axis,
the gradient is not response-specific.

**One deliberate difference from `eph_01`:** no RT trial window is applied
(`trial_query=""`, `min_trials=50`), matching the source notebook and `eph_08`
so the axes fitted here describe the same units `eph_08`'s poster figure
projects. `RT_QUERY` below is `eph_01`'s convention, left available for a
sensitivity check.

### Spike-count windows — read before running

The counts below use a **500 ms post-cue response window and a 2 s pre-cue baseline**.
These are the windows of the run that produced the reference figure: `code/archive/
spatial_axis_comparison_rt_encoding.ipynb` cell 9 (`execution_count` 7) builds
`all_counts_df` with `count_window_s=(0.0, 0.5)`, `baseline_window_s=(-2, 0.0)`, and the
figure is written at `execution_count` 37 of the same run to
`/root/capsule/scratch/figures/poster/rt_response_projection_abs.svg` (2026-05-04,
committed 16 min later in `2f20188`). They also match the published analysis, which uses a
500 ms window after the go cue and a 2 s pre-cue baseline.

They are **not** the `eph_01`-`eph_06` windows (0-200 ms response, -1-0 s baseline).
Earlier drafts of this notebook used those, inherited from a commented-out `cfg` in
`spatial_axis_comparison_rt_encoding_update.ipynb` — a file created *after* the figure, in
the same commit, whose commented block had already been changed. With 0-200 ms / -1-0 s the
waveform-axis projection gives Spearman r = 0.013; the reference reports r = 0.184.

**Check after running:** the response window should give **50 nominally significant /
43 FDR significant** units out of 103 (baseline: 44 / 38). If it reports 42 / 36, the
0-200 ms / -1-0 s windows are still in effect.

In [ ]:
if not IS_CO:
    print("[skip] unit loading + RT encoding (Code Ocean only)")
else:
    import pickle
    from aind_dynamic_foraging_behavior_video_analysis.ephys.tongue_ephys import get_session_prefix
    from data_loading import (
        load_session_quality_filter, filter_ephys_units, load_units_with_spike_times,
    )
    from ephys_utils import AnalysisConfig, build_all_counts_df
    from encoding_methods import AnalysisSpec, fit_encoding
    from per_unit_stats_registry import PerUnitStatsRegistry

    # --- units: tongue-QC session filter + unit QC criteria ---
    with open(SCRATCH / "combined_unit_tbl.pkl", "rb") as f:
        combined_ephys_data = pickle.load(f)

    base_dirs = [SCRATCH / "session_analysis_mlk"]
    filtered_session_paths = load_session_quality_filter(base_dirs)
    filtered_ephys = filter_ephys_units(combined_ephys_data, filtered_session_paths)

    # --- trial x unit spike counts, built in-notebook ---
    # Windows are the reference run's, NOT eph_01-eph_06's -- see the note above.
    cfg = AnalysisConfig(
        align_key="goCue",
        count_window_s=(0.0, 0.5),      # 500 ms post-cue response window
        baseline_window_s=(-2.0, 0.0),  # 2 s pre-cue baseline
        min_trials_per_group=20,
    )
    units_with_spikes = load_units_with_spike_times(
        filtered_ephys, str(DATA / "LC-NE_scratch_data_1")
    )
    all_counts_df = build_all_counts_df(units_with_spikes, cfg, base_dirs)
    print("all_counts_df:", all_counts_df.shape)

    # eph_01's RT window, for the sensitivity check noted above (not applied here).
    RT_QUERY = "reaction_time_firstmove > 0.05 and reaction_time_firstmove < 1.0"

    specs = [
        AnalysisSpec(
            name="ols_rt_response",
            predictor_col="reaction_time_firstmove",
            response_col="spike_count",
            method="ols", trial_query="", log_x=True, zscore_x=True, min_trials=50,
            notes="RT axis source: log(RT) z-scored -> spike_count, response window, no RT window",
        ),
        AnalysisSpec(
            name="ols_rt_baseline",
            predictor_col="reaction_time_firstmove",
            response_col="baseline_spike_count",
            method="ols", trial_query="", log_x=True, zscore_x=True, min_trials=50,
            notes="Control axis: same predictor in the baseline window",
        ),
    ]

    reg = PerUnitStatsRegistry(get_session_prefix=get_session_prefix, alpha=0.05)
    for spec in specs:
        reg.register(fit_encoding(all_counts_df, spec))
    print(reg)

    # Window check -- see the note above.
    for nm in ("ols_rt_response", "ols_rt_baseline"):
        tb = reg.get(nm)
        print(f"  {nm}: {len(tb)} units, {int((tb['p'] < 0.05).sum())} nominally sig, "
              f"{int(tb['sig_fdr'].sum())} FDR sig")
    print("  reference (500 ms / 2 s windows): response 50 / 43, baseline 44 / 38")

In [ ]:
if not IS_CO:
    print("[skip] merge RT stats onto CCF coordinates (Code Ocean only)")
else:
    def canon_unit(x):
        """Match the registry's unit key formatting (e.g. 12.0 -> '12')."""
        try:
            return str(int(float(x)))
        except Exception:
            return str(x)

    features_combined = filtered_ephys.copy()
    features_combined["unit_str"] = features_combined["unit"].map(canon_unit)

    # T_rt / T_rt_bl are the registry t-statistics keyed by (session_prefix, unit).
    for out_col, reg_name in [("T_rt", "ols_rt_response"), ("T_rt_bl", "ols_rt_baseline")]:
        tbl = (reg.get(reg_name)[["session_prefix", "unit", "t"]]
               .rename(columns={"unit": "unit_str", "t": out_col}))
        features_combined = features_combined.merge(
            tbl, on=["session_prefix", "unit_str"], how="left")

    ccf_ok = features_combined[["x_ccf", "y_ccf", "z_ccf"]].notna().all(axis=1)
    print(f"Units: {len(features_combined)}")
    print(f"  with T_rt    + CCF coords: {int((features_combined['T_rt'].notna() & ccf_ok).sum())}")
    print(f"  with T_rt_bl + CCF coords: {int((features_combined['T_rt_bl'].notna() & ccf_ok).sum())}")

## 3. CCF coordinate frame and the LC mesh

Coordinates are converted to bregma-centered LPS mm and folded onto the left
hemisphere, so units recorded on either side contribute to one gradient. The LC
mesh supplies both the arrow origin (its centroid) and the outlines drawn behind
the arrows in §7.

In [ ]:
if not IS_CO:
    print("[skip] CCF setup + LC mesh (Code Ocean only)")
else:
    from trimesh import load_mesh
    from ccf_utils import pir_to_lps, project_to_plane

    def ccf_points_lps_mm(df, fold_left=True):
        """Nx3 bregma-centered LPS mm coords; optionally fold ML to the left."""
        ccfs = df[["x_ccf", "y_ccf", "z_ccf"]].to_numpy(dtype=float) - BREGMA_LPS_MM
        if fold_left:
            ccfs[:, ml] = -np.abs(ccfs[:, ml])
        return ccfs

    MESH_PATH = (DATA / "LC-NE_scratch_data_1" / "combined" / "ccf_maps"
                 / "20250418_transformed_remesh_10_ccf25.obj")
    mesh = load_mesh(str(MESH_PATH))
    mesh_verts_pir_mm = (np.array(mesh.vertices, dtype=float) - BREGMA_PIR_VOX) * (CCF_RES_UM / 1000.0)
    mesh_verts_lps_mm = pir_to_lps(mesh_verts_pir_mm)

    mesh_contours = {
        name: project_to_plane(mesh_verts_lps_mm, axes, pitch=0.02, margin=0.5)
        for name, axes in PLANES.items()
    }

    mesh_centroid = np.mean(mesh_verts_lps_mm, axis=0)
    mesh_centroid[ml] = -np.abs(mesh_centroid[ml])  # left-fold, to match unit coords
    print(f"Mesh contours computed for {list(mesh_contours)}")
    print(f"Mesh centroid (LPS mm, left-folded): {np.round(mesh_centroid, 3)}")

## 4. Fit the RT-encoding spatial axis

`T_rt` is one scalar per unit, so the axis is the direction of the OLS gradient
of `T_rt` over `(ML, AP, DV)` — the direction in which RT encoding changes
fastest. The bootstrap resamples units to get a confidence cone around it.

**This is the step `eph_08` skipped**: `eph_08` projects `|T_rt|` onto the
*waveform* axis but never fits the RT axis, so until now there was nothing to
compare the structural axes *against*.

The 95% cone half-angle is the number to read first — a wide cone (say > 45°)
means the direction is barely determined and every comparison in §6 is weak,
regardless of the p-values.

In [ ]:
if not IS_CO:
    print("[skip] RT encoding spatial axis (Code Ocean only)")
else:
    mask_rt = (features_combined["T_rt"].notna()
               & features_combined[["x_ccf", "y_ccf", "z_ccf"]].notna().all(axis=1))
    fc_rt = features_combined.loc[mask_rt].copy()

    coords_rt  = ccf_points_lps_mm(fc_rt, fold_left=True)
    feature_rt = fc_rt["T_rt"].to_numpy(dtype=float)
    print(f"Fitting spatial axis for RT encoding: n = {len(feature_rt)} units")

    res_rt = bootstrap_spatial_axis_linear(
        feature_rt, coords_rt, n_boot=N_BOOT, seed=42, align_to_observed=True,
    )
    axis_rt      = res_rt["axis_unit"]
    axis_boot_rt = res_rt["axis_boot"]

    ha_rt, _ = cone_half_angle(axis_rt, axis_boot_rt, q=95)
    print(f"RT encoding spatial axis: {np.round(axis_rt, 4)}")
    print(f"  beta (raw):          {np.round(res_rt['beta'], 4)}")
    print(f"  95% cone half-angle: {ha_rt:.1f} deg")
    print(f"  Bootstrap: {res_rt['n_boot_valid']} valid / {res_rt['n_boot_failed']} failed")

In [ ]:
if not IS_CO:
    print("[skip] baseline RT encoding spatial axis (Code Ocean only)")
else:
    mask_bl = (features_combined["T_rt_bl"].notna()
               & features_combined[["x_ccf", "y_ccf", "z_ccf"]].notna().all(axis=1))
    fc_bl = features_combined.loc[mask_bl].copy()

    coords_bl  = ccf_points_lps_mm(fc_bl, fold_left=True)
    feature_bl = fc_bl["T_rt_bl"].to_numpy(dtype=float)

    res_bl = bootstrap_spatial_axis_linear(
        feature_bl, coords_bl, n_boot=N_BOOT, seed=43, align_to_observed=True,
    )
    axis_bl      = res_bl["axis_unit"]
    axis_boot_bl = res_bl["axis_boot"]

    ha_bl, _ = cone_half_angle(axis_bl, axis_boot_bl, q=95)
    print(f"Baseline RT encoding spatial axis: {np.round(axis_bl, 4)}  (n = {len(feature_bl)})")
    print(f"  95% cone half-angle: {ha_bl:.1f} deg")
    print(f"  Bootstrap: {res_bl['n_boot_valid']} valid / {res_bl['n_boot_failed']} failed")

## 5. Structural axes

Three independent descriptions of how LC is organized in space, all from the upstream
capsule. Each gets its own guarded block: if the asset is missing the block
prints why, leaves its `HAS_*` flag False, and every later section simply omits
that axis.

| Axis | Feature per location | Method |
|---|---|---|
| Waveform | 7 spike-waveform shape features per unit | CCA |
| MERFISH | gene expression per cell (2k-gene panel) | CCA |
| Retrograde | injection region label per labelled cell | LDA |

CCA suits the first two because the feature is multivariate: it finds the
spatial direction most correlated with *some* combination of features. The
retrograde data is a categorical label, so the axis is the linear discriminant
that best separates projection classes in space.

### 5a. Waveform features (CCA)

In [ ]:
if not IS_CO:
    print("[skip] waveform CCA axis (Code Ocean only)")
else:
    # The waveform-feature CSV is a SEPARATELY ATTACHED data asset, not part of
    # LC-NE_scratch_data_1 -- the archived older version of this analysis hardcoded
    # "results-59472bbb-4c3a-40f9-a1f5-b0c5113e4ab9-waveforms_np". The `_update` source
    # rewrote that to FIG_PREP_DIR/waveforms_np, which is the upstream layout, not this capsule's.
    # Try both, then glob, so a re-attached asset with a different id still resolves.
    def find_wf_features_csv(data_root):
        """Locate the waveform combined_features.csv across known/likely mount points."""
        cands = [
            data_root / "LC-NE_scratch_data_1" / "combined" / "waveforms_np" / "combined_features.csv",
            data_root / "results-59472bbb-4c3a-40f9-a1f5-b0c5113e4ab9-waveforms_np" / "combined_features.csv",
        ]
        cands += sorted(data_root.glob("*waveforms_np*/combined_features.csv"))
        cands += sorted(data_root.glob("**/combined_features.csv"))
        seen = set()
        for c in cands:
            if c in seen:
                continue
            seen.add(c)
            if c.exists():
                return c
        raise FileNotFoundError(
            "combined_features.csv not found under %s. Tried:\n  %s"
            % (data_root, "\n  ".join(str(c) for c in seen))
        )

    FIG_PREP_DIR = DATA / "LC-NE_scratch_data_1" / "combined"
    try:
        with open(FIG_PREP_DIR / "combine_unit_tbl" / "combined_unit_tbl.pkl", "rb") as f:
            han_units = pickle.load(f)
        wf_csv = find_wf_features_csv(DATA)
        print(f"waveform features: {wf_csv}")
        wf_feats = pd.read_csv(wf_csv)
        wf_feats = wf_feats.merge(
            han_units[["session", "unit", "x_ccf", "y_ccf", "z_ccf"]],
            on=["session", "unit"], how="left",
        )

        wf_feature_cols = [
            "post_w", "trough_post_ratio_1D", "post_trough_slope", "pre_slope",
            "symmetry_slope_div_log", "symmetry_trough_dis", "symmetry_inte_div_log",
        ]

        ccf_wf = wf_feats[["x_ccf", "y_ccf", "z_ccf"]].values - BREGMA_LPS_MM
        ccf_wf[:, ml] = np.abs(ccf_wf[:, ml])
        valid_wf = (~np.any(np.isnan(ccf_wf), axis=1)
                    & ~np.any(np.isnan(wf_feats[wf_feature_cols].values), axis=1))

        res_wf = bootstrap_spatial_axis_cca(
            wf_feats[wf_feature_cols].values[valid_wf], ccf_wf[valid_wf],
            n_boot=N_BOOT, seed=16, align_to_observed=True,
        )
        axis_wf      = res_wf["axis_unit"]
        axis_boot_wf = res_wf["axis_boot"]

        ha_wf, _ = cone_half_angle(axis_wf, axis_boot_wf)
        print(f"Waveform axis: {np.round(axis_wf, 4)}, cone={ha_wf:.1f} deg, "
              f"n={int(valid_wf.sum())}, canonical r={res_wf['canonical_corr']:.3f}")
        HAS_WAVEFORM = True

    except Exception as e:
        print(f"Could not load waveform data: {e}")
        print("Skipping the waveform axis; it will be omitted downstream.")
        HAS_WAVEFORM = False

### 5b. MERFISH spatial transcriptomics (CCA)

Counts are library-size normalized, then the most ventral cells (spatial
`y > 220`) are dropped — they fall outside LC proper. Requires `scanpy`.

In [ ]:
if not IS_CO:
    print("[skip] MERFISH CCA axis (Code Ocean only)")
else:
    try:
        import scanpy as sc
        from ccf_utils import ccf_pts_convert_to_mm

        merfish_path = DATA / "merfish_data" / "adata" / "adata_mer_subset_2_2k.h5ad"
        adata_mer = sc.read_h5ad(str(merfish_path))

        # Library-size normalize
        librarysize = np.sum(adata_mer.X, 1)
        adata_mer.X = 1000 * adata_mer.X / librarysize[:, None]

        # Remove ventral cells (outside LC)
        y_coord = adata_mer.obsm["spatial"][:, 1]
        adata_mer = adata_mer[y_coord <= 220].copy()

        features_mer = adata_mer.X
        coord_mer_mm = ccf_pts_convert_to_mm(adata_mer.obsm["spatial"])
        coord_mer_mm = pir_to_lps(coord_mer_mm)
        coord_mer_mm[:, ml] = np.abs(coord_mer_mm[:, ml])
        coord_mer_mm[:, ap] = -coord_mer_mm[:, ap]

        valid_mer    = coord_mer_mm[:, dv] > -10
        features_mer = features_mer[valid_mer]
        coord_mer_mm = coord_mer_mm[valid_mer]

        res_mer = bootstrap_spatial_axis_cca(
            features_mer, coord_mer_mm, n_boot=N_BOOT, seed=2, align_to_observed=True,
        )
        axis_mer      = res_mer["axis_unit"]
        axis_boot_mer = res_mer["axis_boot"]

        ha_mer, _ = cone_half_angle(axis_mer, axis_boot_mer)
        print(f"MERFISH axis: {np.round(axis_mer, 4)}, cone={ha_mer:.1f} deg, "
              f"n={len(coord_mer_mm)}, canonical r={res_mer['canonical_corr']:.3f}")
        HAS_MERFISH = True

    except Exception as e:
        print(f"Could not load MERFISH data: {e}")
        print("Skipping the MERFISH axis; it will be omitted downstream. "
              "(Needs scanpy — see TODO.md if the import is what failed.)")
        HAS_MERFISH = False

### 5c. Retrograde tracing (LDA)

Thalamic (`TH`) injections are excluded, as in the source. **The sign of an LDA
axis is arbitrary** — it depends on class ordering, so it can flip between runs
and would flip the arrow in §7. The source pins it by aligning to the waveform
axis, and that convention is kept here; when the waveform axis is unavailable
the sign is left as LDA returned it and the arrow direction is not meaningful
(the *angles* in §6 are unaffected either way).

In [ ]:
if not IS_CO:
    print("[skip] retrograde LDA axis (Code Ocean only)")
else:
    try:
        retro_ccf = pd.read_csv(DATA / "LC_retro" / "manual_proofread_ccf_18brains.csv")

        bregma_pixel_retro = np.array([228, 18, 216])
        ccf_retro = (retro_ccf[["x", "y", "z"]].values - bregma_pixel_retro) * 25 / 1000
        ccf_retro[:, [dv, ap]] = ccf_retro[:, [ap, dv]]
        ccf_retro[:, dv] = -ccf_retro[:, dv]
        ccf_retro[:, ml] = np.abs(ccf_retro[:, ml])

        valid_retro   = (retro_ccf["injection_region"].values != "TH") & (ccf_retro[:, dv] > -5)
        feature_retro = retro_ccf["injection_region"].values[valid_retro]
        coord_retro   = ccf_retro[valid_retro]

        res_retro = bootstrap_spatial_axis_LDA(
            feature_retro, coord_retro, n_boot=N_BOOT, seed=0, align_to_observed=True,
        )

        # Pin the arbitrary LDA sign to the waveform axis (source convention).
        if HAS_WAVEFORM and np.dot(res_retro["axis_unit"], axis_wf) < 0:
            res_retro["axis_unit"] = -res_retro["axis_unit"]
            res_retro["axis_boot"] = -res_retro["axis_boot"]
            print("Retrograde LDA sign flipped to align with the waveform axis.")
        elif not HAS_WAVEFORM:
            print("No waveform axis available — retrograde LDA sign left arbitrary.")

        axis_retro      = res_retro["axis_unit"]
        axis_boot_retro = res_retro["axis_boot"]

        ha_retro, _ = cone_half_angle(axis_retro, axis_boot_retro)
        print(f"Retrograde axis: {np.round(axis_retro, 4)}, cone={ha_retro:.1f} deg, "
              f"n={len(coord_retro)}, classes={sorted(set(feature_retro))}")
        HAS_RETRO = True

    except Exception as e:
        print(f"Could not load retrograde data: {e}")
        print("Skipping the retrograde axis; it will be omitted downstream.")
        HAS_RETRO = False

## 6. Pairwise axis comparison

Every axis against every other. Both directions and their bootstrap clouds are
projected into the tangent plane at the first axis; the Wald statistic asks
whether the observed offset is large relative to the paired bootstrap spread.

In [ ]:
if not IS_CO:
    print("[skip] pairwise axis comparison (Code Ocean only)")
else:
    axes_dict = {
        "RT encoding": (axis_rt, axis_boot_rt),
        "RT baseline": (axis_bl, axis_boot_bl),
    }
    if HAS_WAVEFORM:
        axes_dict["Waveform (CCA)"] = (axis_wf, axis_boot_wf)
    if HAS_MERFISH:
        axes_dict["MERFISH (CCA)"] = (axis_mer, axis_boot_mer)
    if HAS_RETRO:
        axes_dict["Retrograde (LDA)"] = (axis_retro, axis_boot_retro)

    missing = [n for n, ok in [("Waveform", HAS_WAVEFORM), ("MERFISH", HAS_MERFISH),
                               ("Retrograde", HAS_RETRO)] if not ok]
    print(f"Axes in play ({len(axes_dict)}): {list(axes_dict)}")
    if missing:
        print(f"OMITTED (asset unavailable): {missing} — comparisons below exclude them.")

    print("=" * 70)
    print("PAIRWISE SPATIAL AXIS COMPARISONS")
    print("=" * 70)

    comparison_results = {}
    for (name_a, (ax_a, boot_a)), (name_b, (ax_b, boot_b)) in combinations(axes_dict.items(), 2):
        n_min = min(len(boot_a), len(boot_b))  # pair bootstrap draws 1:1
        res = compare_bootstrap_directions(ax_a, ax_b, boot_a[:n_min], boot_b[:n_min])
        comparison_results[(name_a, name_b)] = res

        print(f"\n{name_a}  vs.  {name_b}")
        print(f"  Angle:    {res['angle_deg']:.1f} deg")
        print(f"  Wald W:   {res['W_obs']:.2f}")
        print(f"  p (chi2): {res['p_chi2']:.4g}")
        print(f"  p (boot): {res['p_boot']:.4g}")
        if res["p_boot"] > 0.05:
            print("  -> NOT significantly different (axes may be aligned)")
        else:
            print("  -> Significantly DIFFERENT axes")

### Interpretation guide

Read the angle and the p-value **together** — neither alone says much.

| Angle | p-value | Interpretation |
|-------|---------|----------------|
| Small (< 20°) | > 0.05 | Axes are **aligned** — RT encoding varies along the same spatial gradient as the structural feature |
| Large (> 45°) | < 0.05 | Axes are **different** — RT encoding has its own spatial organization |
| Intermediate | depends | Partial overlap; the axes share some but not all spatial structure |

If RT encoding aligns with the waveform / MERFISH / retrograde axis, it suggests
that the same dorsoventral (or other) organizational axis that segregates LC cell
types also organizes functional RT encoding — connecting the partial-correlation
finding in `eph_03` to the structural organization of LC.

Two failure modes the table alone will not show you:

- **Small angle, significant p.** Tight bootstraps can resolve a genuine but
  small offset. The axes are close; they are not the same. Read the cone
  half-angles from §4–§5 before calling this a dissociation.
- **Large angle, non-significant p.** Usually means one cone is wide — the data
  cannot pin that direction down, so it is compatible with almost anything.
  That is an absence of evidence, not evidence of independence.

The `RT baseline` row is the control: an RT axis that differs from the structural
axes is only interesting if the *baseline* axis does not behave the same way.

In [ ]:
if not IS_CO:
    print("[skip] comparison summary table (Code Ocean only)")
else:
    rows = []
    for (name_a, name_b), res in comparison_results.items():
        rows.append({
            "Axis A": name_a,
            "Axis B": name_b,
            "Angle (deg)": f"{res['angle_deg']:.1f}",
            "Wald W": f"{res['W_obs']:.2f}",
            "p (chi2)": f"{res['p_chi2']:.4g}",
            "p (boot)": f"{res['p_boot']:.4g}",
            "Aligned?": "yes" if res["p_boot"] > 0.05 else "no",
        })

    summary_df = pd.DataFrame(rows)
    print(summary_df.to_string(index=False))

## 7. Visualize the axes

### 7a. Projected arrows with 95% confidence cones

Each axis is drawn from the LC mesh centroid on three anatomical planes, with
its bootstrap cone. **A cone is drawn after projection**, so an axis pointing
out of the displayed plane looks short and its cone looks wide — compare cone
widths within a panel, not across panels, and check the numeric half-angles
printed in §4–§5.

In [ ]:
if not IS_CO:
    print("[skip] projected-arrow cone figure (Code Ocean only)")
else:
    fig, axes_arr = plt.subplots(1, 3, figsize=(15, 5))

    for plane, ax in zip(PLANES.keys(), axes_arr):
        for c in mesh_contours[plane]:
            ax.fill(c[:, 0], c[:, 1], color="lightgray", alpha=0.3, linewidth=0)

        dims = PLANES[plane]
        origin = np.mean(mesh_verts_lps_mm[:, [dims[0], dims[1]]], axis=0)
        if ml in dims:  # left-fold the origin to match the folded coordinates
            origin[dims.index(ml)] = -np.abs(origin[dims.index(ml)])

        for name, (ax_vec, boot_vec) in axes_dict.items():
            plot_projected_arrow_with_cone(
                ax, origin, ax_vec, boot_vec, dims,
                color=COLORS.get(name, "gray"), scale=0.8,
                head_width=0.06, head_length=0.10,
                cone_q=95, cone_alpha=0.15, label=name,
            )

        ax.set_title(f"{plane} plane")
        ax.set_xlabel(AXIS_LABEL[dims[0]])
        ax.set_ylabel(AXIS_LABEL[dims[1]])
        ax.set_aspect("equal")
        style_ax(ax)

    axes_arr[-1].legend(fontsize=7, loc="best", frameon=False)
    fig.suptitle("Spatial axis comparison: RT encoding vs. structural organization", y=1.02)
    plt.tight_layout()
    save_fig(fig, "structural_axes_cones_3plane", fig_dir=FIG_DIR, save=SAVE_FIG)
    plt.show()

### 7b. Bootstrap clouds in azimuth–elevation

The same information without the projection artifact: every bootstrap direction
as a point on the sphere, flattened to azimuth and elevation, with the observed
axis marked (×). Overlapping clouds mean indistinguishable axes; a tight cluster
means a well-determined direction.

Elevation compresses near ±90°, so clouds close to the poles look wider than
they are.

In [ ]:
if not IS_CO:
    print("[skip] azimuth-elevation scatter (Code Ocean only)")
else:
    fig, ax = plt.subplots(figsize=(7, 7))

    for name, (ax_vec, boot_vec) in axes_dict.items():
        color = COLORS.get(name, "gray")
        az, el = vectors_to_az_el(boot_vec)
        ax.scatter(az, el, s=3, alpha=0.5, color=color, edgecolor="none", label=name)

        az_obs, el_obs = vectors_to_az_el(np.asarray(ax_vec).reshape(1, 3))
        ax.scatter(az_obs, el_obs, s=150, marker="x", color=color, linewidths=2.5, zorder=10)

    ax.set_xlabel("Azimuth (deg)")
    ax.set_ylabel("Elevation (deg)")
    ax.set_title("Bootstrap spatial axis distributions (x = observed axis)")
    ax.set_aspect("equal")
    ax.legend(fontsize=9)
    style_ax(ax)
    plt.tight_layout()
    save_fig(fig, "structural_axes_azimuth_elevation", fig_dir=FIG_DIR, save=SAVE_FIG)
    plt.show()

### Anatomical filter: `z_ccf` bounds

The reference run excludes units whose CCF dorsoventral coordinate falls outside the LC
range `z_ccf` ∈ [-5.2, -3.5] (LPS mm). On the current unit table this drops exactly one
unit — `behavior_758017_2025-02-06_11-26-14` unit 85, `z_ccf = -2.174`, which sits 2.25 mm
from the LC mesh centroid while every other unit is within 0.79 mm — taking the projection
from n = 100 to the reference's n = 99.

Unit QC in `data_loading` filters on spike-sorting quality only (`isi_violations`, `p_max`,
`snr`, `qc_pass`, ...) and never checks anatomy, so this is the only step that removes
mislocalised units.

**Placement matters.** The reference applies this filter *after* fitting the spatial
axes and *before* the projection scatters: its axis fit reports `n = 100 units`
(`execution_count` 14) while the projections report `n=99` (`execution_count` 30-37).
Keep this cell here, immediately before §8, or §4's axes will be fitted on 99 units
and will no longer match the reference.

The cell is destructive on `features_combined`: re-running it after the filter has applied
reports 0 units outside, which is expected, not a failure. (The reference notebook's stored
output shows exactly that — it reports 0 outside yet n falls from 100 to 99, because the
cell had already been run once.)

In [ ]:
if not IS_CO:
    print("[skip] z_ccf anatomical filter (Code Ocean only)")
else:
    Z_CCF_BOUNDS = (-5.2, -3.5)   # LC dorsoventral range, bregma-relative LPS mm

    outside = (features_combined["z_ccf"].notna()
               & ~features_combined["z_ccf"].between(*Z_CCF_BOUNDS))
    print(f"z_ccf outside {Z_CCF_BOUNDS}: {int(outside.sum())} unit(s)")
    if outside.any():
        print(features_combined.loc[outside, ["session", "unit", "z_ccf"]]
              .to_string(index=False))

    features_combined = features_combined.loc[
        features_combined["z_ccf"].isna()
        | features_combined["z_ccf"].between(*Z_CCF_BOUNDS)
    ].copy()

    ccf_ok = features_combined[["x_ccf", "y_ccf", "z_ccf"]].notna().all(axis=1)
    print(f"Units after filter: {len(features_combined)}")
    print(f"  with T_rt    + CCF coords: "
          f"{int((features_combined['T_rt'].notna() & ccf_ok).sum())}   (reference: 99)")
    print(f"  with T_rt_bl + CCF coords: "
          f"{int((features_combined['T_rt_bl'].notna() & ccf_ok).sum())}   (reference: 99)")

## 8. Projection scatters: RT encoding along each structural gradient

§6 compares axis *directions*. This section asks the per-unit version of the
same question: project each unit's CCF coordinate onto a structural axis to get
its scalar position along that gradient, then correlate that position with the
unit's RT-encoding t-statistic.

A significant correlation means units at different points along the structural
gradient differ in how strongly they encode RT — the unit-level evidence behind
an alignment. Coordinates are centered on the mesh centroid first, so projection
0 is the middle of LC rather than bregma.

`T_rt_bl` is the control: it should be *less* structured than `T_rt` if the
gradient is specific to the response window.

In [ ]:
if not IS_CO:
    print("[skip] structural-axis projection setup (Code Ocean only)")
else:
    structural_axes, structural_colors = {}, {}

    if HAS_WAVEFORM:
        structural_axes["wf_axis"] = axis_wf
        structural_colors["wf_axis"] = COLORS["Waveform (CCA)"]
    if HAS_MERFISH:
        structural_axes["merfish_axis"] = axis_mer
        structural_colors["merfish_axis"] = COLORS["MERFISH (CCA)"]
    if HAS_RETRO:
        structural_axes["retro_axis"] = axis_retro
        structural_colors["retro_axis"] = COLORS["Retrograde (LDA)"]

    if not structural_axes:
        print("No structural axes loaded - skipping the projection scatters below.")
    else:
        print(f"Structural axes available: {list(structural_axes)}")
        print(f"Centering projections on mesh centroid: {np.round(mesh_centroid, 3)}")


def sig_stars(p):
    """Conventional significance marker for a p-value."""
    if not np.isfinite(p):
        return "n/a"
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"

### 8a. RT response encoding (`T_rt`) projected onto the structural axes

In [ ]:
if not IS_CO:
    print("[skip] T_rt projection scatter (Code Ocean only)")
elif not structural_axes:
    print("[skip] T_rt projection scatter (no structural axes available)")
else:
    fc_proj     = features_combined.loc[mask_rt]
    coords_proj = ccf_points_lps_mm(fc_proj, fold_left=True)
    feat_proj   = fc_proj["T_rt"].to_numpy(dtype=float)

    fig_rt, results_rt = plot_projection_scatter(
        coords_proj, feat_proj, "T_rt (response)",
        structural_axes, structural_colors,
        center_coords_on_mesh=mesh_centroid,
    )
    save_fig(fig_rt, "T_rt_projection_on_structural_axes", fig_dir=FIG_DIR, save=SAVE_FIG)
    plt.show()

    print("\nRT response encoding projected onto structural axes:")
    for name, res in results_rt.items():
        print(f"  {name}: r={res['r']:.3f}, p={res['p']:.4g}, n={res['n']}  {sig_stars(res['p'])}")

### 8b. RT baseline encoding (`T_rt_bl`) projected onto the structural axes

In [ ]:
if not IS_CO:
    print("[skip] T_rt_bl projection scatter (Code Ocean only)")
elif not structural_axes:
    print("[skip] T_rt_bl projection scatter (no structural axes available)")
else:
    fc_proj_bl     = features_combined.loc[mask_bl]
    coords_proj_bl = ccf_points_lps_mm(fc_proj_bl, fold_left=True)
    feat_proj_bl   = fc_proj_bl["T_rt_bl"].to_numpy(dtype=float)

    fig_bl, results_bl = plot_projection_scatter(
        coords_proj_bl, feat_proj_bl, "T_rt_bl (baseline)",
        structural_axes, structural_colors,
        center_coords_on_mesh=mesh_centroid,
    )
    save_fig(fig_bl, "T_rt_bl_projection_on_structural_axes", fig_dir=FIG_DIR, save=SAVE_FIG)
    plt.show()

    print("\nRT baseline encoding projected onto structural axes:")
    for name, res in results_bl.items():
        print(f"  {name}: r={res['r']:.3f}, p={res['p']:.4g}, n={res['n']}  {sig_stars(res['p'])}")

### 8c. Combined projection summary

In [ ]:
if not IS_CO:
    print("[skip] projection summary table (Code Ocean only)")
elif not structural_axes:
    print("[skip] projection summary table (no structural axes available)")
else:
    rows = []
    for feat_name, feat_results in [("T_rt (response)", results_rt),
                                    ("T_rt_bl (baseline)", results_bl)]:
        for axis_name, res in feat_results.items():
            rows.append({
                "Feature": feat_name,
                "Structural axis": axis_name,
                "Spearman r": f"{res['r']:.3f}",
                "p-value": f"{res['p']:.4g}",
                "n": res["n"],
                "sig": sig_stars(res["p"]),
            })

    proj_summary = pd.DataFrame(rows)
    print(proj_summary.to_string(index=False))
    print()
    print("Interpretation: a significant correlation means units at different positions")
    print("along the structural gradient (e.g. dorsal vs ventral LC as defined by waveform")
    print("type) differ in how strongly they encode RT. Compare the T_rt rows against the")
    print("T_rt_bl rows - a gradient present in both is not response-specific.")